This notebook removes certain features or groups of features from a model to measure how much they contribute to its performance.

The goal is to determine which features are actually useful for making predictions.

- Weather: Temperature, humidity, wind speed, precipitation
- Vegetation: NDVI, EVI (vegetation indexes)
- Topography: Elevation, slope, aspect

| Experiment | Features | Number |
|---|---|---|
| A — Weather only | Temperature, humidity, wind speed, precipitation | 4 |
| B — Weather + vegetation | Weather + NDVI, EVI | 6 |
| C — Full model (already trained) | Weather + vegetation + elevation, slope, aspect | 9 |

Reader notes: First four cells are copied from random_forest notebook not much going on, just ensuring exact parameters

In [14]:
import numpy as np
import pandas as pd
import duckdb
import joblib
import json
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
import time
import gc

from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [6]:
data_path = Path("../data/processed/master_training_dataset.parquet")

if not data_path.is_file():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

print("Dataset found:", data_path)
print("File size (GB):", round(data_path.stat().st_size / 1e9, 2))

# Connect to DuckDB
con = duckdb.connect()

# Inspect dataset columns and data types
schema = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [str(data_path)]
).df()

display(schema)

Dataset found: ../data/processed/master_training_dataset.parquet
File size (GB): 1.13


,column_name,column_type,null,key,default,extra
0,cell_id,INTEGER,YES,None,None,None
1,valid_time,TIMESTAMP_NS,YES,None,None,None
2,number,FLOAT,YES,None,None,None
3,t2m,FLOAT,YES,None,None,None
4,rh2m,FLOAT,YES,None,None,None
5,wind_speed_10m,FLOAT,YES,None,None,None
6,tp,FLOAT,YES,None,None,None
7,ndvi,FLOAT,YES,None,None,None
8,evi,FLOAT,YES,None,None,None
9,longitude,FLOAT,YES,None,None,None


In [7]:
query = """
WITH daily AS (
    SELECT
        cell_id,
        CAST(valid_time AS DATE) AS weather_date,
        t2m,
        rh2m,
        wind_speed_10m,
        tp,
        ndvi,
        evi,
        elevation,
        slope,
        aspect,
        fire_occurred
    FROM read_parquet(?)
    WHERE t2m IS NOT NULL
),

next_day AS (
    SELECT
        a.cell_id,
        a.weather_date,
        b.weather_date AS valid_time,

        a.t2m,
        a.rh2m,
        a.wind_speed_10m,
        a.tp,
        a.ndvi,
        a.evi,
        a.elevation,
        a.slope,
        a.aspect,

        b.fire_occurred
    FROM daily a
    INNER JOIN daily b
        ON a.cell_id = b.cell_id
        AND b.weather_date = a.weather_date + INTERVAL 1 DAY
    WHERE b.weather_date < '2024-01-01'
),

sampled AS (
    SELECT *
    FROM next_day
    WHERE EXTRACT(YEAR FROM valid_time) = 2023
       OR fire_occurred = 1
       OR (
           hash(
               CAST(cell_id AS VARCHAR) || ':' ||
               CAST(valid_time AS VARCHAR)
           ) % 100
       ) < 2
)

SELECT *
FROM sampled
ORDER BY valid_time, cell_id
"""

df = con.execute(
    query,
    [str(data_path)]
).df()

df["valid_time"] = pd.to_datetime(df["valid_time"])
df["weather_date"] = pd.to_datetime(df["weather_date"])

print("Dataset shape:", df.shape)
print("\nClass distribution:")
print(df["fire_occurred"].value_counts())

print("\nDuplicate rows:",
      df.duplicated(["cell_id", "valid_time"]).sum())

display(df.head())

Dataset shape: (28209909, 13)

Class distribution:
fire_occurred
0.0    28202633
1.0        7276
Name: count, dtype: int64

Duplicate rows: 0


,cell_id,weather_date,valid_time,t2m,rh2m,wind_speed_10m,tp,ndvi,evi,elevation,slope,aspect,fire_occurred
0,409,2022-01-01,2022-01-02,261.865540,78.001411,3.082888,0.000035,NaN,NaN,659,89.464157,236.830109,0.0
1,431,2022-01-01,2022-01-02,261.035492,78.330627,3.854872,0.000039,NaN,NaN,576,89.524826,211.778717,0.0
2,451,2022-01-01,2022-01-02,259.924133,79.400047,5.193607,0.000064,NaN,NaN,566,89.130798,54.260223,0.0
3,478,2022-01-01,2022-01-02,259.790375,76.972786,3.345769,0.000065,0.229196,0.109765,232,88.100578,40.292149,0.0
4,522,2022-01-01,2022-01-02,262.139954,77.055847,2.198470,0.000185,0.212067,0.106262,54,35.264389,315.000000,0.0


In [8]:
features = [
    "t2m",
    "rh2m",
    "wind_speed_10m",
    "tp",
    "ndvi",
    "evi",
    "elevation",
    "slope",
    "aspect"
]

# Split by target date
train = df[df["valid_time"].dt.year == 2022].copy()
val = df[df["valid_time"].dt.year == 2023].copy()

X_train = train[features]
y_train = train["fire_occurred"].astype(int)

X_val = val[features]
y_val = val["fire_occurred"].astype(int)

print("Training observations:", len(train))
print("Validation observations:", len(val))

print("\nTraining fire distribution:")
print(y_train.value_counts())

print("\nValidation fire distribution:")
print(y_val.value_counts())

print("\nTraining missing-value rates:")
print(X_train.isna().mean())

assert y_train.nunique() == 2, "Training set needs both classes"
assert y_val.nunique() == 2, "Validation set needs both classes"
assert not df.duplicated(["cell_id", "valid_time"]).any()

Training observations: 554224
Validation observations: 27655685

Training fire distribution:
fire_occurred
0    550616
1      3608
Name: count, dtype: int64

Validation fire distribution:
fire_occurred
0    27652017
1        3668
Name: count, dtype: int64

Training missing-value rates:
t2m               0.000000
rh2m              0.000000
wind_speed_10m    0.000000
tp                0.000000
ndvi              0.035939
evi               0.035939
elevation         0.000000
slope             0.000000
aspect            0.000000
dtype: float64


In [9]:
# Train Random Forest Ablation Models

weather_features = [
    "t2m", "rh2m", "wind_speed_10m", "tp"
]

vegetation_features = [
    "ndvi", "evi"
]

feature_groups = {
    "weather_only": weather_features,
    "weather_vegetation": weather_features + vegetation_features
}

# Same weights used in the original Random Forest experiment
sample_weights = np.where(
    y_train.to_numpy() == 1,
    1.0,
    50.0
)

ablation_models = {}

for experiment, selected_features in feature_groups.items():

    print(f"\nTraining: {experiment}")
    print("Features:", selected_features)

    start_time = time.time()

    # Fit imputer only on training observations
    imputer = SimpleImputer(strategy="median")

    X_train_subset = imputer.fit_transform(
        X_train[selected_features]
    )

    # Match the original Random Forest configuration
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=10,
        max_features="sqrt",
        random_state=42,
        n_jobs=4
    )

    model.fit(
        X_train_subset,
        y_train,
        sample_weight=sample_weights
    )

    ablation_models[experiment] = {
        "model": model,
        "imputer": imputer,
        "features": selected_features
    }

    print(
        f"Training completed in "
        f"{time.time() - start_time:.1f} seconds"
    )

print("\nBoth ablation models trained successfully!")


Training: weather_only
Features: ['t2m', 'rh2m', 'wind_speed_10m', 'tp']
Training completed in 26.8 seconds

Training: weather_vegetation
Features: ['t2m', 'rh2m', 'wind_speed_10m', 'tp', 'ndvi', 'evi']
Training completed in 29.6 seconds

Both ablation models trained successfully!


In [11]:
# Evaluate Ablation Models on 2023 Validation

outputs_dir = Path("../outputs")
models_dir = Path("../models")

outputs_dir.mkdir(exist_ok=True)
models_dir.mkdir(exist_ok=True)

batch_size = 250_000
ablation_results = []

for experiment, bundle in ablation_models.items():
    print(f"\nEvaluating: {experiment}")

    model = bundle["model"]
    imputer = bundle["imputer"]
    selected_features = bundle["features"]

    # Save model for reproducibility
    joblib.dump(
        bundle,
        models_dir / f"rf_ablation_{experiment}.joblib"
    )

    output_path = (
        outputs_dir / f"rf_ablation_{experiment}_validation.parquet"
    )

    writer = None

    try:
        for start in range(0, len(X_val), batch_size):
            end = min(start + batch_size, len(X_val))

            X_batch = imputer.transform(
                X_val.iloc[start:end][selected_features]
            )

            probabilities = model.predict_proba(X_batch)[:, 1]

            batch = val.iloc[start:end][
                ["cell_id", "weather_date", "valid_time"]
            ].copy()

            batch["y_true"] = y_val.iloc[start:end].to_numpy()
            batch["y_prob"] = probabilities
            batch["model"] = experiment

            table = pa.Table.from_pandas(
                batch, preserve_index=False
            )

            if writer is None:
                writer = pq.ParquetWriter(
                    str(output_path),
                    table.schema,
                    compression="zstd"
                )

            writer.write_table(table)

    finally:
        if writer is not None:
            writer.close()

    # Evaluate from saved probabilities
    results_df = pd.read_parquet(
        output_path, columns=["y_true", "y_prob"]
    )

    pr_auc = average_precision_score(
        results_df["y_true"],
        results_df["y_prob"]
    )

    roc_auc = roc_auc_score(
        results_df["y_true"],
        results_df["y_prob"]
    )

    ablation_results.append({
        "model": experiment,
        "features": len(selected_features),
        "pr_auc": float(pr_auc),
        "roc_auc": float(roc_auc)
    })

    print(f"PR-AUC: {pr_auc:.6f}")
    print(f"ROC-AUC: {roc_auc:.6f}")

    del results_df

ablation_df = pd.DataFrame(ablation_results)

# Previously evaluated full-feature Random Forest
full_rf = {
    "model": "full_model",
    "features": 9,
    "pr_auc": 0.4493,
    "roc_auc": 0.9922
}

ablation_df = pd.concat(
    [ablation_df, pd.DataFrame([full_rf])],
    ignore_index=True
)

display(ablation_df)

ablation_df.to_csv(
    outputs_dir / "rf_ablation_2023_comparison.csv",
    index=False
)

print("\n2023 ablation evaluation complete!")


Evaluating: weather_only
PR-AUC: 0.000318
ROC-AUC: 0.686909

Evaluating: weather_vegetation
PR-AUC: 0.005945
ROC-AUC: 0.924680


,model,features,pr_auc,roc_auc
0,weather_only,4,0.000318,0.686909
1,weather_vegetation,6,0.005945,0.924680
2,full_model,9,0.449300,0.992200



2023 ablation evaluation complete!


| Feature group | PR-AUC | ROC-AUC |
|---|---:|---:|
| Weather only | 0.000318 | 0.6869 |
| Weather + vegetation | 0.005945 | 0.9247 |
| All nine features | 0.4493 | 0.9922 |

Weather alone weak, adding vegetation helps proportionally alot, terrain is big improvement

In [12]:
# 2024 Ablation Test Predictions

query_2024 = """
WITH weather AS (
    SELECT
        cell_id,
        CAST(valid_time AS DATE) AS weather_date,
        t2m, rh2m, wind_speed_10m, tp, ndvi, evi
    FROM read_parquet(?)
    WHERE t2m IS NOT NULL
      AND valid_time >= '2023-12-31'
      AND valid_time < '2024-12-31'
),
targets AS (
    SELECT
        cell_id,
        CAST(valid_time AS DATE) AS valid_time,
        fire_occurred
    FROM read_parquet(?)
    WHERE t2m IS NOT NULL
      AND valid_time >= '2024-01-01'
      AND valid_time < '2025-01-01'
)
SELECT w.*, t.valid_time, t.fire_occurred
FROM weather w
INNER JOIN targets t
    ON w.cell_id = t.cell_id
    AND t.valid_time = w.weather_date + INTERVAL 1 DAY
"""

con = duckdb.connect()

reader = con.sql(
    query_2024,
    params=[str(data_path), str(data_path)]
).to_arrow_reader(batch_size=100_000)

writers = {}
test_paths = {
    name: outputs_dir / f"rf_ablation_{name}_test_2024.parquet"
    for name in ablation_models
}

total_rows = 0

try:
    for record_batch in reader:
        batch = record_batch.to_pandas()

        for name, bundle in ablation_models.items():
            X_batch = bundle["imputer"].transform(
                batch[bundle["features"]]
            )

            probabilities = bundle["model"].predict_proba(
                X_batch
            )[:, 1]

            result = batch[
                ["cell_id", "weather_date", "valid_time"]
            ].copy()

            result["y_true"] = batch["fire_occurred"].to_numpy()
            result["y_prob"] = probabilities
            result["model"] = name

            table = pa.Table.from_pandas(
                result, preserve_index=False
            )

            if name not in writers:
                writers[name] = pq.ParquetWriter(
                    str(test_paths[name]),
                    table.schema,
                    compression="zstd"
                )

            writers[name].write_table(table)

        total_rows += len(batch)

        if total_rows % 1_000_000 < len(batch):
            print(f"Processed {total_rows:,} observations")

finally:
    for writer in writers.values():
        writer.close()
    con.close()

assert total_rows == 27_731_454, (
    f"Unexpected 2024 observation count: {total_rows}"
)

print("\n2024 ablation predictions complete!")
print("Total observations:", total_rows)

Processed 1,000,000 observations
Processed 2,000,000 observations
Processed 3,000,000 observations
Processed 4,000,000 observations
Processed 5,000,000 observations
Processed 6,000,000 observations
Processed 7,000,000 observations
Processed 8,000,000 observations
Processed 9,000,000 observations
Processed 10,000,000 observations
Processed 11,000,000 observations
Processed 12,000,000 observations
Processed 13,000,000 observations
Processed 14,000,000 observations
Processed 15,000,000 observations
Processed 16,000,000 observations
Processed 17,000,000 observations
Processed 18,000,000 observations
Processed 19,000,000 observations
Processed 20,000,000 observations
Processed 21,000,000 observations
Processed 22,000,000 observations
Processed 23,000,000 observations
Processed 24,000,000 observations
Processed 25,099,424 observations
Processed 26,068,335 observations
Processed 27,068,862 observations

2024 ablation predictions complete!
Total observations: 27731454


In [15]:
# Evaluate 2024 Ablation Results
test_results = []

for name, path in test_paths.items():
    print(f"\nEvaluating {name}...")

    df = pd.read_parquet(
        path,
        columns=["y_true", "y_prob"]
    )

    y_true = df["y_true"].to_numpy()
    y_prob = df["y_prob"].to_numpy()

    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)

    test_results.append({
        "model": name,
        "features": len(ablation_models[name]["features"]),
        "pr_auc": float(pr_auc),
        "roc_auc": float(roc_auc),
        "observations": int(len(df)),
        "fire_observations": int(y_true.sum())
    })

    print(f"PR-AUC: {pr_auc:.6f}")
    print(f"ROC-AUC: {roc_auc:.6f}")

    del df, y_true, y_prob
    gc.collect()

# Load exact results from the original full-feature RF test
with open(outputs_dir / "rf_test_2024_metrics.json") as f:
    full_metrics = json.load(f)

test_results.append({
    "model": "full_model",
    "features": 9,
    "pr_auc": float(full_metrics["pr_auc"]),
    "roc_auc": float(full_metrics["roc_auc"]),
    "observations": int(full_metrics["observations"]),
    "fire_observations": int(full_metrics["fire_observations"])
})

ablation_2024_df = pd.DataFrame(test_results)

# Ensure all feature groups used the same test population
assert ablation_2024_df["observations"].nunique() == 1
assert ablation_2024_df["fire_observations"].nunique() == 1

display(ablation_2024_df)

ablation_2024_df.to_csv(
    outputs_dir / "rf_ablation_2024_comparison.csv",
    index=False
)

print("\n2024 ablation evaluation complete!")


Evaluating weather_only...
PR-AUC: 0.000579
ROC-AUC: 0.713415

Evaluating weather_vegetation...
PR-AUC: 0.010267
ROC-AUC: 0.923825


,model,features,pr_auc,roc_auc,observations,fire_observations
0,weather_only,4,0.000579,0.713415,27731454,3843
1,weather_vegetation,6,0.010267,0.923825,27731454,3843
2,full_model,9,0.445782,0.990964,27731454,3843



2024 ablation evaluation complete!


### Feature Ablation — Summary

- Purpose: Evaluated how weather, vegetation, and topographic features contribute to Random Forest wildfire ignition prediction.


- **Weather only (4 features):** Achieved 2024 PR-AUC of 0.000579, indicating limited predictive ability using atmospheric variables alone.


- **Weather + vegetation (6 features):** Improved 2024 PR-AUC to 0.010267, showing that NDVI and EVI provide substantial additional predictive information.


- **Full model (9 features):** Achieved 2024 PR-AUC of 0.445782, substantially outperforming the reduced-feature models.


- The same performance ranking was observed on 2023 validation and 2024 testing.

Conclusion: Vegetation and terrain features are strongly associated with improved predictive performance in the processed dataset, with the full-feature Random Forest achieving the best results.